# Лабораторная работа №3: Оптимизация гиперпараметров

## Выбор задачи и модели

В данной работе используется задача классификации пассажиров Титаника (лабораторная №2).

Выбранная модель: **Random Forest** — в лабораторной №2 она показала лучший результат среди ансамблей (Accuracy = 0.8156, F1 = 0.7273).

Исходные параметры (из лабораторной №2):
- n_estimators = 200
- min_samples_leaf = 3
- random_state = 42
- остальные параметры по умолчанию

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set(style="whitegrid")

In [2]:
# Загрузка данных (аналогично лабораторной №2)
data = pd.read_csv("Titanic-Dataset.csv")

# Выбор признаков
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = data[features]
y = data["Survived"]

# Разделение данных с сохранением соотношения классов
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [3]:
# Pipeline для предобработки (аналогично лабораторной №2)
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_features = ["Pclass", "Sex", "Embarked"]
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## Baseline — исходная модель

In [4]:
# Исходная модель из лабораторной №2 (до оптимизации)
baseline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=3,
        random_state=RANDOM_STATE
    ))
])

baseline_rf.fit(X_train, y_train)
baseline_pred = baseline_rf.predict(X_test)

baseline_test_f1 = f1_score(y_test, baseline_pred)
baseline_test_acc = accuracy_score(y_test, baseline_pred)

In [5]:
print(f"Baseline (исходная модель):")
print(f"  Test Accuracy: {baseline_test_acc:.4f}")
print(f"  Test F1:       {baseline_test_f1:.4f}")

## Гиперпараметры для оптимизации

Выбраны 4 значимых гиперпараметра алгоритма Random Forest:

| Параметр | Смысл | Диапазон | Влияние |
|----------|-------|----------|----------|
| `n_estimators` | Количество деревьев в лесу | [50, 100, 200, 300, 500] | Больше деревьев → стабильнее результат, но дольше обучение. После определённого порога прирост минимален. |
| `max_depth` | Максимальная глубина каждого дерева | [3, 5, 7, 10, None] | Малая глубина → недообучение (модель слишком простая). Большая/без ограничения → переобучение. |
| `min_samples_split` | Минимальное число объектов для разделения узла | [2, 5, 10, 20] | Увеличение → регуляризация, модель становится проще, риск переобучения снижается. |
| `min_samples_leaf` | Минимальное число объектов в листе | [1, 2, 3, 5, 10] | Аналогично: больше → более гладкие предсказания, меньше переобучение. |

**Предположения:**
- На Титанике всего 891 объект (712 в трейне), поэтому большая глубина деревьев и маленькие листья приведут к переобучению.
- Оптимальная глубина, вероятно, около 5–10.
- Слишком мало деревьев (50) может дать нестабильный результат, слишком много (500) — не даст значимого прироста.

## Кросс-валидация

Используем **5-кратную стратифицированную кросс-валидацию** (`StratifiedKFold(n_splits=5)`).

**Выбор 5 folds обоснован:**
- Выборка небольшая (712 объектов), 10 фолдов оставили бы слишком мало объектов для обучения (~641) и валидации (~71) в каждом фолде.
- 5 фолдов — стандартный выбор для небольших датасетов, даёт хороший баланс между вычислительной стоимостью и надёжностью оценки.
- Стратификация важна, так как классы несбалансированы (61.6% погибших, 38.4% выживших).

**Ответы на вопросы:**

1. **Зачем нужна кросс-валидация?**
Кросс-валидация позволяет использовать все данные и для обучения, и для оценки модели (каждый объект побывает в валидации ровно один раз). Это даёт более надёжную оценку качества модели и снижает зависимость от конкретного случайного разбиения на train/val.

2. **Почему одного разбиения train/test недостаточно для подбора параметров?**
Одно разбиение создаёт лишь одну точку оценки. Модель может случайно хорошо или плохо работать именно на этом разбиении. При подборе параметров по одному разбиению мы рискуем выбрать параметры, которые хорошо работают только на этом конкретном подмножестве. Кросс-валидация усредняет результат по нескольким разбиениям, что снижает дисперсию оценки.

3. **Что произойдёт, если многократно выбирать гиперпараметры по результату на test?**
Это приведёт к **утечке данных** (data leakage) и **переобучению на тест**. При многократной проверке мы неявно подбираем параметры, которые хорошо работают именно на тестовых данных, а не обобщаются. Финальная метрика на тесте станет оптимистично завышенной и не будет отражать реальное качество модели на новых данных.

In [6]:
# Стратифицированная 5-кратная кросс-валидация для обучения на трейне.
# Test НЕ участвует в подборе гиперпараметров!

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Количество фолдов: {cv.get_n_splits()}")
print(f"Объектов в каждом фолде валидации: ~{X_train.shape[0] // 5}")

## Grid Search

In [7]:
# Определение пространства поиска для GridSearchCV.
# Для управляемости используем уменьшенную сетку.

param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__min_samples_leaf': [1, 2, 3, 5, 10]
}

# Подсчёт комбинаций: 3 * 5 * 4 * 5 = 300 комбинаций.
# Каждая проверяется в 5 фолдах → 1500 обучений модели.
total_combinations = 3 * 5 * 4 * 5
print(f"Пространство поиска: {total_combinations} комбинаций")
print(f"Общее количество обучений: {total_combinations * 5}")

In [8]:
pipeline_for_grid = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
])

grid_search = GridSearchCV(
    estimator=pipeline_for_grid,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

print("Запуск Grid Search...")
start_time = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - start_time
print(f"Grid Search завершён за {grid_time:.2f} секунд")

In [9]:
print("="*60)
print("Результаты Grid Search")
print("="*60)
print(f"Лучшие гиперпараметры: {grid_search.best_params_}")
print(f"Лучший CV F1:          {grid_search.best_score_:.4f}")
print(f"Количество проверенных комбинаций: {len(grid_search.cv_results_['params'])}")
print(f"Время выполнения: {grid_time:.2f} сек")

In [10]:
# Топ-5 комбинаций по CV F1
results_grid = pd.DataFrame(grid_search.cv_results_)
results_grid_sorted = results_grid.sort_values('rank_test_score').head(5)
print("Топ-5 комбинаций (Grid Search):")
print(results_grid_sorted[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']].to_string())

## Random Search

Для сопоставимого вычислительного бюджета используем `n_iter=300` (столько же итераций, сколько комбинаций в Grid Search).

In [11]:
# Пространство поиска для RandomizedSearchCV (можно использовать распределения)
param_distributions = {
    'classifier__n_estimators': [50, 100, 150, 200, 300, 400, 500],
    'classifier__max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 12, 15, None],
    'classifier__min_samples_split': [2, 3, 5, 7, 10, 15, 20, 25],
    'classifier__min_samples_leaf': [1, 2, 3, 4, 5, 7, 10, 15]
}

pipeline_for_random = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
])

random_search = RandomizedSearchCV(
    estimator=pipeline_for_random,
    param_distributions=param_distributions,
    n_iter=300,
    cv=cv,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

print("Запуск Random Search...")
start_time = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start_time
print(f"Random Search завершён за {random_time:.2f} секунд")

In [12]:
print("="*60)
print("Результаты Random Search")
print("="*60)
print(f"Лучшие гиперпараметры: {random_search.best_params_}")
print(f"Лучший CV F1:          {random_search.best_score_:.4f}")
print(f"Количество проверенных комбинаций: {len(random_search.cv_results_['params'])}")
print(f"Время выполнения: {random_time:.2f} сек")

In [13]:
# Топ-5 комбинаций по CV F1 (Random Search)
results_random = pd.DataFrame(random_search.cv_results_)
results_random_sorted = results_random.sort_values('rank_test_score').head(5)
print("Топ-5 комбинаций (Random Search):")
print(results_random_sorted[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']].to_string())

## Сравнение Grid Search и Random Search

In [14]:
comparison = pd.DataFrame({
    'Метод': ['Grid Search', 'Random Search'],
    'Лучший CV F1': [grid_search.best_score_, random_search.best_score_],
    'Кол-во комбинаций': [len(grid_search.cv_results_['params']), len(random_search.cv_results_['params'])],
    'Время (сек)': [round(grid_time, 2), round(random_time, 2)]
})
print(comparison.to_string(index=False))

**Когда случайный поиск предпочтительнее полного перебора?**

1. **Высокая размерность пространства.** Когда гиперпараметров много, полная сетка растёт экспоненциально. Случайный поиск при фиксированном бюджете итераций исследует пространство разнообразнее.

2. **Неравномерная важность параметров.** Как показали Бергстра и Кутрат (2012), часто некоторые гиперпараметры почти не влияют на результат. Полный перебор тратит ресурсы на вариацию неважных параметров, а случайный поиск пробует больше значений важных.

3. **Непрерывные параметры.** Для непрерывных диапазонов (например, `learning_rate`) полная сетка дискретизирует их грубо, а случайный поиск может пробовать значения из непрерывного распределения.

4. **Ограниченный вычислительный бюджет.** При равном времени/числе итераций случайный поиск часто находит решение не хуже или лучше.

## Влияние гиперпараметров на качество модели

In [15]:
# Исследуем влияние каждого гиперпараметра на CV F1,
# фиксируя остальные параметры на лучших значениях из Grid Search.

best_params = grid_search.best_params_
best_n_estimators = best_params['classifier__n_estimators']
best_max_depth = best_params['classifier__max_depth']
best_min_samples_split = best_params['classifier__min_samples_split']
best_min_samples_leaf = best_params['classifier__min_samples_leaf']

print(f"Лучшие параметры (зафиксированы):")
print(f"  n_estimators: {best_n_estimators}")
print(f"  max_depth: {best_max_depth}")
print(f"  min_samples_split: {best_min_samples_split}")
print(f"  min_samples_leaf: {best_min_samples_leaf}")

In [16]:
def evaluate_param(param_name, param_values, fixed_params, cv_strategy):
    """
    Оценивает модель на разных значениях одного гиперпараметра,
    фиксируя остальные на лучших значениях.
    """
    train_scores = []
    test_scores = []
    test_stds = []
    
    for val in param_values:
        params = fixed_params.copy()
        params[f'classifier__{param_name}'] = val
        
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
        ])
        pipe.set_params(**params)
        
        # CV оценка (на трейне)
        cv_results = cross_val_score(pipe, X_train, y_train, cv=cv_strategy, scoring='f1')
        test_scores.append(cv_results.mean())
        test_stds.append(cv_results.std())
        
        # Train оценка (для определения переобучения)
        pipe.fit(X_train, y_train)
        train_pred = pipe.predict(X_train)
        train_scores.append(f1_score(y_train, train_pred))
    
    return param_values, train_scores, test_scores, test_stds

In [17]:
# Фиксированные параметры (лучшие из Grid Search)
fixed = {
    'classifier__n_estimators': best_n_estimators,
    'classifier__max_depth': best_max_depth,
    'classifier__min_samples_split': best_min_samples_split,
    'classifier__min_samples_leaf': best_min_samples_leaf
}

# 1. Влияние max_depth
depth_values = [2, 3, 4, 5, 6, 7, 8, 10, 12, 15, None]
d_vals, d_train, d_test, d_std = evaluate_param('max_depth', depth_values, fixed, cv)

In [18]:
# 2. Влияние n_estimators
est_values = [10, 25, 50, 100, 150, 200, 300, 400, 500]
e_vals, e_train, e_test, e_std = evaluate_param('n_estimators', est_values, fixed, cv)

In [19]:
# 3. Влияние min_samples_leaf
leaf_values = [1, 2, 3, 5, 7, 10, 15, 20, 30]
l_vals, l_train, l_test, l_std = evaluate_param('min_samples_leaf', leaf_values, fixed, cv)

In [20]:
# 4. Влияние min_samples_split
split_values = [2, 3, 5, 7, 10, 15, 20, 30, 50]
s_vals, s_train, s_test, s_std = evaluate_param('min_samples_split', split_values, fixed, cv)

In [21]:
# Визуализация влияния гиперпараметров (4 подграфика)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# max_depth
depth_labels = [str(v) if v is not None else 'None' for v in d_vals]
axes[0, 0].plot(range(len(depth_labels)), d_train, 'o-', label='Train F1', color='darkorange')
axes[0, 0].plot(range(len(depth_labels)), d_test, 's-', label='CV F1', color='steelblue')
axes[0, 0].fill_between(range(len(depth_labels)),
                         np.array(d_test) - np.array(d_std),
                         np.array(d_test) + np.array(d_std), alpha=0.2, color='steelblue')
axes[0, 0].set_xticks(range(len(depth_labels)))
axes[0, 0].set_xticklabels(depth_labels, rotation=45)
axes[0, 0].set_xlabel('max_depth')
axes[0, 0].set_ylabel('F1 Score')
axes[0, 0].set_title('Влияние max_depth')
axes[0, 0].legend()

# n_estimators
axes[0, 1].plot(e_vals, e_train, 'o-', label='Train F1', color='darkorange')
axes[0, 1].plot(e_vals, e_test, 's-', label='CV F1', color='steelblue')
axes[0, 1].fill_between(e_vals, np.array(e_test) - np.array(e_std),
                         np.array(e_test) + np.array(e_std), alpha=0.2, color='steelblue')
axes[0, 1].set_xlabel('n_estimators')
axes[0, 1].set_ylabel('F1 Score')
axes[0, 1].set_title('Влияние n_estimators')
axes[0, 1].legend()

# min_samples_leaf
axes[1, 0].plot(l_vals, l_train, 'o-', label='Train F1', color='darkorange')
axes[1, 0].plot(l_vals, l_test, 's-', label='CV F1', color='steelblue')
axes[1, 0].fill_between(l_vals, np.array(l_test) - np.array(l_std),
                         np.array(l_test) + np.array(l_std), alpha=0.2, color='steelblue')
axes[1, 0].set_xlabel('min_samples_leaf')
axes[1, 0].set_ylabel('F1 Score')
axes[1, 0].set_title('Влияние min_samples_leaf')
axes[1, 0].legend()

# min_samples_split
axes[1, 1].plot(s_vals, s_train, 'o-', label='Train F1', color='darkorange')
axes[1, 1].plot(s_vals, s_test, 's-', label='CV F1', color='steelblue')
axes[1, 1].fill_between(s_vals, np.array(s_test) - np.array(s_std),
                         np.array(s_test) + np.array(s_std), alpha=0.2, color='steelblue')
axes[1, 1].set_xlabel('min_samples_split')
axes[1, 1].set_ylabel('F1 Score')
axes[1, 1].set_title('Влияние min_samples_split')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

**Анализ графиков:**

**max_depth:**
- При малой глубине (2–3) модель **недообучается**: и train F1, и CV F1 низкие, разрыв между ними мал.
- Оптимальная глубина находится в области 5–10: CV F1 максимальный.
- При большой/неограниченной глубине (12, 15, None) train F1 растёт, а CV F1 падает или стабилизируется — признак **переобучения**. Разрыв между кривыми увеличивается.

**n_estimators:**
- При малом числе деревьев (10–50) результат нестабильный.
- После 100–200 деревьев прирост минимален. Это типичное поведение для бэггинга: усреднение большого числа моделей снижает дисперсию, но после порога эффект насыщения.
- Переобучения по числу деревьев практически нет: и train, и CV ведут себя стабильно.

**min_samples_leaf:**
- При min_samples_leaf=1 модель переобучается (высокий train, ниже CV, большой разрыв).
- Увеличение до 3–5 улучшает обобщение.
- При слишком больших значениях (20–30) модель становится слишком простой → недообучение.

**min_samples_split:**
- Аналогично: малые значения → переобучение, большие → недообучение.
- Оптимум в районе 5–15 для данного датасета.

## Финальная модель и оценка на тестовой выборке

Тестовая выборка использовалась **только один раз** — для финальной оценки уже выбранных моделей. Все решения о параметрах принимались по кросс-валидации на трейне.

In [22]:
# Выбираем лучшую модель между Grid и Random по CV.
if grid_search.best_score_ >= random_search.best_score_:
    best_search = 'Grid Search'
    best_search_model = grid_search
else:
    best_search = 'Random Search'
    best_search_model = random_search

print(f"Лучший метод поиска: {best_search} (CV F1 = {best_search_model.best_score_:.4f})")

In [23]:
# Обучаем финальную модель с лучшими параметрами и оцениваем на тесте.
# Используем best_estimator_ — он уже обучен на всём трейне.

# Оценка на тесте для лучших моделей из Grid и Random.
grid_test_pred = grid_search.predict(X_test)
random_test_pred = random_search.predict(X_test)

grid_test_f1 = f1_score(y_test, grid_test_pred)
grid_test_acc = accuracy_score(y_test, grid_test_pred)

random_test_f1 = f1_score(y_test, random_test_pred)
random_test_acc = accuracy_score(y_test, random_test_pred)

print("="*60)
print("Финальные результаты на тестовой выборке")
print("="*60)
print(f"Grid Search:   Test F1 = {grid_test_f1:.4f}, Test Accuracy = {grid_test_acc:.4f}")
print(f"Random Search: Test F1 = {random_test_f1:.4f}, Test Accuracy = {random_test_acc:.4f}")

In [24]:
# Итоговая таблица сравнения трёх моделей.
final_comparison = pd.DataFrame({
    'Модель': ['Исходная (из лаб.2)', 'Лучшая после Grid Search', 'Лучшая после Random Search'],
    'CV F1': [None, grid_search.best_score_, random_search.best_score_],
    'Test F1': [baseline_test_f1, grid_test_f1, random_test_f1],
    'Test Accuracy': [baseline_test_acc, grid_test_acc, random_test_acc]
})

# Для исходной модели посчитаем CV.
cv_baseline = cross_val_score(baseline_rf, X_train, y_train, cv=cv, scoring='f1')
final_comparison.loc[0, 'CV F1'] = cv_baseline.mean()

print(final_comparison.round(4).to_string(index=False))

**Ответы на вопросы:**

1. **Насколько оптимизация гиперпараметров улучшила качество?**
Сравниваем CV F1 исходной модели и оптимизированных. Улучшение может быть небольшим (например, 0.01–0.04), так как датасет маленький и исходные параметры из лабораторной №2 уже были подобраны достаточно хорошо.

2. **Совпадает ли улучшение на кросс-валидации с результатом на test?**
Не обязательно. Если на CV модель показала прирост, но на test разницы нет или она меньше — это говорит о высокой дисперсии оценки из-за малого размера тестовой выборки (всего 179 объектов). Если на test результат лучше, чем на CV — модель могла «подстроиться» под особенности конкретного теста.

3. **Если нет, чем это можно объяснить?**
- Малый размер датасета: тестовая выборка всего 179 объектов, метрика на ней имеет высокую дисперсию.
- Разница в распределении объектов между фолдами и финальным тестом.
- Кросс-валидация оценивает обобщение усреднённо по 5 фолдам, а тест — одна конкретная выборка.

## Анализ устойчивости (стабильность по фолдам)

In [25]:
# Детальные результаты по каждому фолду для лучшей модели.
best_cv_results = pd.DataFrame(best_search_model.cv_results_)
best_row = best_cv_results.loc[best_cv_results['rank_test_score'].idxmin()]

print(f"Лучшая модель ({best_search}):")
print(f"Параметры: {best_row['params']}")
print(f"\nF1 по фолдам:")
for i in range(5):
    print(f"  Fold {i+1}: {best_row[f'split{i}_test_score']:.4f}")
print(f"\n  Mean: {best_row['mean_test_score']:.4f}")
print(f"  Std:  {best_row['std_test_score']:.4f}")

In [26]:
# Сравнение устойчивости: исходная модель против оптимизированной.
cv_scores_baseline = cross_val_score(baseline_rf, X_train, y_train, cv=cv, scoring='f1')

# Для лучшей модели извлекаем результаты по фолдам.
cv_scores_best = np.array([best_row[f'split{i}_test_score'] for i in range(5)])

stability_df = pd.DataFrame({
    'Модель': ['Исходная', 'Оптимизированная'],
    'Mean F1': [cv_scores_baseline.mean(), cv_scores_best.mean()],
    'Std F1': [cv_scores_baseline.std(), cv_scores_best.std()],
    'Min F1': [cv_scores_baseline.min(), cv_scores_best.min()],
    'Max F1': [cv_scores_baseline.max(), cv_scores_best.max()],
    'Размах (Max-Min)': [cv_scores_baseline.max() - cv_scores_baseline.min(),
                          cv_scores_best.max() - cv_scores_best.min()]
})
print(stability_df.round(4).to_string(index=False))

In [27]:
# Визуализация распределения по фолдам (боксплот).
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([cv_scores_baseline, cv_scores_best],
                labels=['Исходная', 'Оптимизированная'])
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Распределение F1 по фолдам')
axes[0].axhline(y=cv_scores_baseline.mean(), color='orange', linestyle='--', alpha=0.5, label='Mean исходной')
axes[0].axhline(y=cv_scores_best.mean(), color='blue', linestyle='--', alpha=0.5, label='Mean оптимизир.')
axes[0].legend()

# Сравнение по фолдам (точечный график).
fold_nums = range(1, 6)
axes[1].plot(fold_nums, cv_scores_baseline, 'o-', label='Исходная', color='darkorange')
axes[1].plot(fold_nums, cv_scores_best, 's-', label='Оптимизированная', color='steelblue')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 по фолдам')
axes[1].legend()

plt.tight_layout()
plt.show()

**Почему две модели с одинаковым средним качеством могут различаться по устойчивости?**

Среднее значение метрики по фолдам скрывает дисперсию. Две модели могут иметь одинаковый mean F1, но:
- Одна стабильно показывает 0.72–0.74 на всех фолдах (низкий разброс).
- Другая показывает 0.60 на одном фолде и 0.86 на другом (высокий разброс), но среднее тоже ~0.73.

Вторая модель **менее надёжна**: её качество сильно зависит от конкретного набора данных, попавшего в обучение/валидацию. Это признак того, что модель чувствительна к составу обучающей выборки, что особенно критично на малых данных.

Регуляризация (ограничение глубины, увеличение минимального числа объектов в листе) обычно снижает не только переобучение, но и дисперсию предсказаний между фолдами, повышая устойчивость.

## Дополнительные метрики финальной модели

In [28]:
# Детальные метрики лучшей модели на тестовой выборке.
best_pred = best_search_model.predict(X_test)

print("="*60)
print(f"Детальные метрики лучшей модели ({best_search}) на тесте")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, best_pred):.4f}")
print(f"Precision: {precision_score(y_test, best_pred):.4f}")
print(f"Recall:    {recall_score(y_test, best_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, best_pred):.4f}")

In [29]:
# Confusion Matrix лучшей модели.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=["Погиб", "Выжил"]).plot(cmap='Blues')
plt.title(f"Confusion Matrix ({best_search})")
plt.show()

## Итоговая таблица

In [30]:
# Сводная таблица всех экспериментов.
summary = pd.DataFrame({
    'Модель': ['Исходная (лаб.2)', 'После Grid Search', 'После Random Search'],
    'CV F1 (mean ± std)': [
        f"{cv_baseline.mean():.4f} ± {cv_baseline.std():.4f}",
        f"{grid_search.best_score_:.4f} ± {best_cv_results.loc[grid_search.cv_results_['rank_test_score'].idxmin(), 'std_test_score']:.4f}",
        f"{random_search.best_score_:.4f} ± {best_cv_results.loc[random_search.cv_results_['rank_test_score'].idxmin(), 'std_test_score']:.4f}"
    ],
    'Test F1': [baseline_test_f1, grid_test_f1, random_test_f1],
    'Test Accuracy': [baseline_test_acc, grid_test_acc, random_test_acc],
    'Время поиска (сек)': ['—', round(grid_time, 2), round(random_time, 2)],
    'Комбинаций проверено': ['—', len(grid_search.cv_results_['params']), len(random_search.cv_results_['params'])]
})

print(summary.to_string(index=False))

## Вывод

1. **Оптимизация гиперпараметров позволила улучшить качество модели.** Итоговый CV F1 и тестовый F1 оптимизированной модели превышают или сопоставимы с исходной моделью из лабораторной №2.

2. **Grid Search и Random Search показали сопоставимые результаты.** При одинаковом количестве проверенных комбинаций (300) оба метода нашли близкие по качеству решения. На практике при большей размерности пространства поиска случайный поиск был бы предпочтительнее.

3. **Кросс-валидация позволила надёжно оценить модель.** Использование 5-кратной стратифицированной кросс-валидации на обучающей выборке дало устойчивую оценку качества и предотвратило утечку данных из тестовой выборки.

4. **Анализ влияния гиперпараметров подтвердил ожидаемые закономерности:**
   - Слишком малая глубина → недообучение.
   - Слишком большая глубина → переобучение (разрыв между train и CV).
   - Увеличение числа деревьев стабилизирует результат, но с насыщением.
   - Регуляризация через `min_samples_leaf` и `min_samples_split` помогает снизить переобучение на малом датасете.

5. **Устойчивость модели важна не меньше среднего качества.** Разброс метрики между фолдами показывает, насколько модель зависит от конкретного состава обучающих данных. Оптимизация гиперпараметров, как правило, снижает не только среднюю ошибку, но и дисперсию.

6. **Честный эксперимент соблюдён:** тестовая выборка была использована ровно один раз — для финальной оценки уже выбранных моделей. Все решения о гиперпараметрах принимались исключительно по результатам кросс-валидации на обучающей выборке. Все `random_state` зафиксированы, эксперимент воспроизводим.